# Observation-Sequence Likelihood Calculation Using Hidden Markov Model (HMM)

**Course / Syllabus:** Module 5 — HMM Forward Algorithm  
**Team Members:**
1. `1AT23AI050` — SHIVASHANKAR
2. `1AT23AI051` — SOMOIPTA NANDY
3. `1AT23AI052` — SOMESH CHAUDHARY

---

This notebook demonstrates the **Forward Algorithm** to compute observation sequence likelihood, displays the full **Forward Probability (Alpha) Table**, and verifies the result using the **Backward Algorithm**.

In [1]:
# imports
import numpy as np
from hmm import HiddenMarkovModel

### Build the model

In [2]:
# initialize model parameters (Hot/Cold Weather & Ice Cream Example)
state_space = ["hot", "cold"]
observation_space = [1, 2, 3]
initial_probabilities = [0.8, 0.2]
transition_probabilities = [[0.6, 0.4], [0.5, 0.5]]
emission_probabilities = [[0.2, 0.4, 0.4], [0.5, 0.4, 0.1]]

In [3]:
# build model
hmm = HiddenMarkovModel(state_space, observation_space, transition_probabilities, 
                        emission_probabilities, initial_probabilities)

### Problem 1: Likelihood (Forward Algorithm & Alpha Table)

Given an HMM $\lambda = (A, B, \pi)$ and an observation sequence $O$, determine the likelihood $P(O|\lambda)$.

**Forward Variable Recursion:**
$$\alpha_t(j) = \left[ \sum_{i=1}^N \alpha_{t-1}(i) \cdot a_{ij} \right] \cdot b_j(O_t)$$

**Termination (Final Likelihood):**
$$P(O|\lambda) = \sum_{i=1}^N \alpha_{T-1}(i)$$

In [4]:
# calculate likelihood using forward algorithm
observation_sequence = [1, 2, 3, 2, 2, 1, 2]
prob_fwd, alpha = hmm.likelihood_forward(observation_sequence)

print("* Observation sequence: {}\n".format(observation_sequence))

# display alpha table
T = len(observation_sequence)
col_headers = ["t={} (obs={})".format(t, o) for t, o in enumerate(observation_sequence)]
print("FORWARD PROBABILITY (ALPHA) TABLE:")
print("-" * 100)
print("{:<10}".format("State") + "".join(["{:<14}".format(h) for h in col_headers]))
print("-" * 100)
for i, state in enumerate(state_space):
    print("{:<10}".format(state) + "".join(["{:<14.8f}".format(alpha[i, t]) for t in range(T)]))
print("-" * 100)
print("{:<10}".format("Sum") + "".join(["{:<14.8f}".format(alpha[:, t].sum()) for t in range(T)]))
print("-" * 100)

print("\n* Final Forward Likelihood P(O|lambda): {:.10f} ({:.2e})".format(prob_fwd, prob_fwd))

# verify with backward algorithm
prob_bwd, beta = hmm.likelihood_backward(observation_sequence)
print("\nVERIFICATION (FORWARD vs BACKWARD):\n" + "* Forward Algorithm Likelihood  : {:.12f}".format(prob_fwd))
print("* Backward Algorithm Likelihood : {:.12f}".format(prob_bwd))
print("* Absolute Difference           : {:.2e}".format(abs(prob_fwd - prob_bwd)))
print("* Verification Status           : {}".format("PASSED (Exact Match)" if np.isclose(prob_fwd, prob_bwd) else "FAILED"))

* Observation sequence: [1, 2, 3, 2, 2, 1, 2]

FORWARD PROBABILITY (ALPHA) TABLE:
----------------------------------------------------------------------------------------------------
State       t=0 (obs=1)   t=1 (obs=2)   t=2 (obs=3)   t=3 (obs=2)   t=4 (obs=2)   t=5 (obs=1)   t=6 (obs=2)
----------------------------------------------------------------------------------------------------
hot          0.16000000    0.05840000    0.02313600    0.00647584    0.00247919    0.00049362    0.00031569
cold         0.10000000    0.04560000    0.00461600    0.00462496    0.00196113    0.00098612    0.00027620
----------------------------------------------------------------------------------------------------
Sum          0.26000000    0.10400000    0.02775200    0.01110080    0.00444032    0.00147974    0.00059189
----------------------------------------------------------------------------------------------------

* Final Forward Likelihood P(O|lambda): 0.0005918945 (5.92e-04)

VERIFICATION (FO

### Problem 2: Decoding

Given an observation sequence $O$ and an HMM $\lambda = (A, B, \pi)$, discover the best hidden state sequence $Q$.

In [5]:
# determine most likely state sequence
path, prob = hmm.decode(observation_sequence)
print("* Observation sequence: {}".format(observation_sequence))
print("* Most likely hidden state path: {}".format(path))
print("* Likelihood for observation sequence along path: {:.2e}".format(prob))

* Observation sequence: [1, 2, 3, 2, 2, 1, 2]
* Most likely hidden state path: ['hot' 'hot' 'hot' 'hot' 'hot' 'cold' 'hot']
* Likelihood for observation sequence along path: 2.12e-05


### Problem 3: Learning

Given an observation sequence $O$ and the set of states in the HMM, learn the HMM parameters $A$, $B$, and $\pi$.

In [6]:
# print original parameters
print("PROBABILITIES BEFORE LEARNING")
print("-----------------------------")
print("* Initial:")
print(hmm.pi)
print("\n* Transition:")
print(hmm.tp)
print("\n* Emission:")
print(hmm.ep)

PROBABILITIES BEFORE LEARNING
-----------------------------
* Initial:
[0.8 0.2]

* Transition:
[[0.6 0.4]
 [0.5 0.5]]

* Emission:
[[0.2 0.4 0.4]
 [0.5 0.4 0.1]]


In [7]:
# learn from observation sequence
hmm.learn(observation_sequence, iterations=1)

In [8]:
# print parameters after learning
print("PROBABILITIES AFTER LEARNING")
print("----------------------------")
print("* Initial:")
print(hmm.pi)
print("\n* Transition:")
print(hmm.tp)
print("\n* Emission:")
print(hmm.ep)

PROBABILITIES AFTER LEARNING
----------------------------
* Initial:
[0.61804435 0.38195565]

* Transition:
[[0.60945272 0.39054728]
 [0.50990073 0.49009927]]

* Emission:
[[0.23642616 0.55648474 0.2070891 ]
 [0.35240033 0.59164734 0.05595234]]
